In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_kf_oof = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_kf_oof_train2_index_fixed.parquet')

print(train_kf_oof.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(616293, 40)


In [4]:
# 종속변수 분위수 기준으로 구간 설정
quantiles = [0, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]
labels = ['0~50%', '50~75%', '75~90%', '90~95%', '95~99%', '99~100%']

train_kf_oof['y_kf_val_quant'] = pd.qcut(  # pd.qcut() 지정한 분위수 경계값 기준으로 구간 나눠줌
    train_kf_oof['y_kf_val'],  # 매개변수: 구간 나눌 데이터
    q=quantiles,  # 매개변수: 구간을 나눌 위치(분위수)
    labels=labels # 매개변수: 각각의 라벨(q보다 하나 적어야 함)
)

In [3]:
train_kf_oof.head()

,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,imerg_rain_1h,imerg_rain_6h,...,wl_diff_24h,imerg_int_1to6h,imerg_int_6to24h,sfc_int_1to6h,sfc_int_6to24h,y_kf_val,y_kf_pred,resid,resid_sq,fold_num
row_id,,,,,,,,,,,,,,,,,,,,,
TR_000222241,2100-01-01,ST0001,0.29,0.29,0.28,0.28,0.28,0.27,0.000,0.000,...,0.02,0.000,0.000000,0.0,0.0,0.00,0.024048,-0.024048,0.000578,5
TR_000288748,2100-01-01,ST0002,0.77,0.76,0.77,0.77,0.76,0.77,0.000,0.000,...,0.00,0.000,0.000000,0.0,0.0,0.00,0.003616,-0.003616,0.000013,5
TR_000737576,2100-01-01,ST0003,0.73,0.73,0.74,0.74,0.73,0.73,0.000,0.000,...,0.00,0.000,0.000000,0.0,0.0,0.00,0.014925,-0.014925,0.000223,5
TR_000651952,2100-01-01,ST0004,1.01,1.01,1.01,1.01,1.01,1.01,0.000,0.000,...,0.00,0.000,0.000000,0.0,0.0,0.00,-0.005762,0.005762,0.000033,5
TR_000328380,2100-01-01,ST0005,1.01,1.01,1.01,1.01,1.03,1.06,3.018,7.493,...,-0.05,0.895,0.000222,0.0,0.0,0.03,0.039088,-0.009088,0.000083,5


In [5]:
# ============================================================
# 강우사상 × 종속변수 분위수 교차분석
# ============================================================

# RMSE 상위 6개 강우사상
top6_event = [98, 90, 35, 97, 3, 89]

event_y_quant_analysis = (
    train_kf_oof[
        train_kf_oof['event_id'].isin(top6_event)
    ]
    .groupby(
        ['event_id', 'y_kf_val_quant'],
        observed=True
    )
    .agg(
        count=('resid_sq', 'size'),
        mse=('resid_sq', 'mean'),
        sse=('resid_sq', 'sum')
    )
)

event_y_quant_analysis['rmse'] = (
    event_y_quant_analysis['mse'] ** 0.5
)

display(
    event_y_quant_analysis[
        ['count', 'rmse', 'sse']
    ]
)

count      rmse          sse
event_id y_kf_val_quant                              
3        0~50%            4370  0.152419   101.521296
         50~75%           3934  0.124036    60.524370
         75~90%           5156  0.151672   118.610057
         90~95%           2214  0.227106   114.191593
         95~99%           1828  0.628095   721.151090
         99~100%           557  1.647468  1511.782912
35       0~50%            8447  0.115362   112.416083
         50~75%           3063  0.110957    37.709960
         75~90%           3715  0.166598   103.109046
         90~95%           1865  0.254373   120.676351
         95~99%           2129  0.618369   814.086840
         99~100%           748  1.687826  2130.870530
89       0~50%           13081  0.115615   174.851397
         50~75%           5345  0.115299    71.055526
         75~90%           4134  0.165064   112.636130
         90~95%           1290  0.300580   116.549140
         95~99%           1175  0.706901   587.158795
         99~100%           264  2.153547  1224.369373
90       0~50%           13753  0.098835   134.344184
         50~75%           4274  0.097411    40.555798
         75~90%           3894  0.155517    94.178491
         90~95%           1301  0.299425   116.641531
         95~99%           1334  0.776449   804.231868
         99~100%           414  2.508908  2605.971870
97       0~50%            2272  0.172744    67.797326
         50~75%            744  0.140401    14.666119
         75~90%           1391  0.197279    54.136520
         90~95%            868  0.291558    73.785508
         95~99%           1681  0.819480  1128.871206
         99~100%           452  2.035535  1872.817169
98       0~50%           41515  0.153506   978.259879
         50~75%           8416  0.159898   215.174957
         75~90%           9695  0.222215   478.735864
         90~95%           5244  0.334329   586.152001
         95~99%           5663  0.710680  2860.192884
         99~100%          1034  1.579048  2578.167499

In [7]:
# 강우사상별 전체 SSE 대비 분위수별 SSE 기여율(%)
event_y_quant_analysis['sse_pct'] = (
    event_y_quant_analysis['sse']
    / event_y_quant_analysis.groupby(level='event_id')['sse'].transform('sum')
    * 100
)

# 강우사상 × 종속변수 분위수 : RMSE 비교
display(
    event_y_quant_analysis['rmse'].unstack().round(3)
)

# 강우사상 × 종속변수 분위수 : SSE 기여율 비교
display(
    event_y_quant_analysis['sse_pct'].unstack().round(1)
)

y_kf_val_quant,0~50%,50~75%,75~90%,90~95%,95~99%,99~100%
event_id,,,,,,
3,0.152,0.124,0.152,0.227,0.628,1.647
35,0.115,0.111,0.167,0.254,0.618,1.688
89,0.116,0.115,0.165,0.301,0.707,2.154
90,0.099,0.097,0.156,0.299,0.776,2.509
97,0.173,0.140,0.197,0.292,0.819,2.036
98,0.154,0.160,0.222,0.334,0.711,1.579


y_kf_val_quant,0~50%,50~75%,75~90%,90~95%,95~99%,99~100%
event_id,,,,,,
3,3.9,2.3,4.5,4.3,27.4,57.5
35,3.4,1.1,3.1,3.6,24.5,64.2
89,7.6,3.1,4.9,5.1,25.7,53.5
90,3.5,1.1,2.5,3.1,21.2,68.7
97,2.1,0.5,1.7,2.3,35.1,58.3
98,12.7,2.8,6.2,7.6,37.2,33.5
